[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_1/02_tag_1_regression_mietpreise.ipynb)

# Tag 1 - 02 Regression mit Mietpreisen

Regression bedeutet: Das Modell gibt eine Zahl aus. Hier ist diese Zahl die Monatsmiete. Die einfache Modellform ist eine Gerade:

`Vorhersage = m * x + b`

`m` ist die Steigung, `b` ist der Achsenabschnitt. Beide Werte können manuell gesetzt oder aus Trainingsdaten gelernt werden.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
print('Setup abgeschlossen.')


## Daten und Zielgröße

Eine Zeile ist eine Wohnung. Für den ersten Regressionsschritt nutzen wir vor allem `Wohnflaeche_qm`, weil man daran eine Gerade gut versteht. Weitere Spalten bleiben sichtbar, damit klar ist: In echten Projekten kann `X` mehrere Merkmale enthalten.


In [ ]:
def make_rent_data(n=140, noise=85, outliers=0, outlier_strength=700, seed=RANDOM_STATE):
    local_rng = np.random.default_rng(seed)
    wohnflaeche = local_rng.uniform(28, 155, n)
    zimmer = np.clip(np.round(wohnflaeche / 32 + local_rng.normal(0, 0.6, n)), 1, 6)
    entfernung = local_rng.uniform(0.5, 18.0, n)

    # Die Miete ist synthetisch, aber fachlich plausibel:
    # größere Wohnung teurer, mehr Zimmer teurer, weiter weg vom Zentrum günstiger.
    miete_ohne_rauschen = 320 + 10.5 * wohnflaeche + 115 * zimmer - 24 * entfernung
    miete = miete_ohne_rauschen + local_rng.normal(0, noise, n)

    outlier_idx = np.array([], dtype=int)
    if outliers > 0:
        outlier_idx = local_rng.choice(n, size=min(outliers, n), replace=False)
        miete[outlier_idx] += outlier_strength

    return pd.DataFrame({
        'Wohnflaeche_qm': wohnflaeche,
        'Zimmer': zimmer.astype(int),
        'Entfernung_Zentrum_km': entfernung,
        'Monatsmiete_EUR': miete,
        'Ausreisser': np.isin(np.arange(n), outlier_idx),
    })

def split_train_test(df, test_fraction=0.25, seed=RANDOM_STATE):
    ids = np.random.default_rng(seed).permutation(len(df))
    test_size = int(test_fraction * len(df))
    test_idx = ids[:test_size]
    train_idx = ids[test_size:]
    return df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

example_df = make_rent_data(noise=85, outliers=4)
display(example_df.head())
print('Samples:', len(example_df))
print('Features in X:', ['Wohnflaeche_qm', 'Zimmer', 'Entfernung_Zentrum_km'])
print('Zielgröße y: Monatsmiete_EUR')


## Was macht eine Gerade?

Die Gerade erzeugt für jeden x-Wert eine Vorhersage. Danach wird der Fehler zur echten Miete berechnet. Eine gute Gerade hat kleine Fehler auf Trainingsdaten und bleibt auch auf Testdaten brauchbar.


In [ ]:
def add_bias_column(X):
    X = np.asarray(X, dtype=float)
    return np.c_[np.ones(len(X)), X]

def fit_linear(X, y):
    # Geschlossene Lösung für lineare Regression.
    # Ergebnis: weights[0] ist b, weights[1] ist m.
    return np.linalg.lstsq(add_bias_column(X), y, rcond=None)[0]

def predict_linear(X, weights):
    return add_bias_column(X) @ weights

def predict_manual_line(x_values, m, b):
    return m * np.asarray(x_values, dtype=float) + b

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    errors = y_pred - y_true
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - np.sum(errors ** 2) / ss_tot if ss_tot > 0 else np.nan
    return mae, rmse, r2


## Gerade einstellen und mit gelernter Gerade vergleichen

Stelle `m` und `b` ein. Schwarz ist deine Gerade. Orange ist die Gerade, die aus den Trainingsdaten gelernt wurde. Die Tabelle zeigt, ob die Gerade nur auf Training gut aussieht oder auch auf Testdaten.


In [ ]:
def line_fit_demo(samples=140, m=10.0, b=350.0, rauschen=85, ausreisser=0, ausreisser_staerke=700):
    df = make_rent_data(n=samples, noise=rauschen, outliers=ausreisser, outlier_strength=ausreisser_staerke)
    train_df, test_df = split_train_test(df)

    feature = 'Wohnflaeche_qm'
    learned_weights = fit_linear(train_df[[feature]], train_df['Monatsmiete_EUR'])

    manual_train = predict_manual_line(train_df[feature], m, b)
    manual_test = predict_manual_line(test_df[feature], m, b)
    learned_train = predict_linear(train_df[[feature]], learned_weights)
    learned_test = predict_linear(test_df[[feature]], learned_weights)

    rows = []
    for name, pred_train, pred_test in [
        ('manuell', manual_train, manual_test),
        ('gelernt', learned_train, learned_test),
    ]:
        train_mae, train_rmse, train_r2 = regression_metrics(train_df['Monatsmiete_EUR'], pred_train)
        test_mae, test_rmse, test_r2 = regression_metrics(test_df['Monatsmiete_EUR'], pred_test)
        rows.append({
            'Modell': name,
            'Train Ausreißer': int(train_df['Ausreisser'].sum()),
            'Test Ausreißer': int(test_df['Ausreisser'].sum()),
            'Train MAE': train_mae,
            'Train RMSE': train_rmse,
            'Train R2': train_r2,
            'Test MAE': test_mae,
            'Test RMSE': test_rmse,
            'Test R2': test_r2,
        })
    display(pd.DataFrame(rows).round(2))

    grid = np.linspace(df[feature].min(), df[feature].max(), 250)
    manual_grid = predict_manual_line(grid, m, b)
    learned_grid = predict_linear(pd.DataFrame({feature: grid}), learned_weights)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].scatter(train_df.loc[~train_df['Ausreisser'], feature], train_df.loc[~train_df['Ausreisser'], 'Monatsmiete_EUR'], alpha=0.45, label='Training normal')
    axes[0].scatter(test_df.loc[~test_df['Ausreisser'], feature], test_df.loc[~test_df['Ausreisser'], 'Monatsmiete_EUR'], alpha=0.85, label='Test normal')
    if train_df['Ausreisser'].any():
        axes[0].scatter(train_df.loc[train_df['Ausreisser'], feature], train_df.loc[train_df['Ausreisser'], 'Monatsmiete_EUR'], marker='x', s=100, color='red', label='Ausreißer Training')
    if test_df['Ausreisser'].any():
        axes[0].scatter(test_df.loc[test_df['Ausreisser'], feature], test_df.loc[test_df['Ausreisser'], 'Monatsmiete_EUR'], marker='D', s=70, color='purple', label='Ausreißer Test')
    axes[0].plot(grid, manual_grid, color='black', linewidth=2, label=f'manuell: y={m:.1f}x+{b:.0f}')
    axes[0].plot(grid, learned_grid, color='orange', linewidth=2, label=f'gelernt: y={learned_weights[1]:.1f}x+{learned_weights[0]:.0f}')
    axes[0].set_xlabel('Wohnfläche in qm')
    axes[0].set_ylabel('Monatsmiete in EUR')
    axes[0].set_title('Daten und Geraden')
    axes[0].legend()

    manual_test_metrics = regression_metrics(test_df['Monatsmiete_EUR'], manual_test)
    learned_test_metrics = regression_metrics(test_df['Monatsmiete_EUR'], learned_test)
    axes[1].bar(['manuell MAE', 'manuell RMSE', 'gelernt MAE', 'gelernt RMSE'], [manual_test_metrics[0], manual_test_metrics[1], learned_test_metrics[0], learned_test_metrics[1]], color=['black', 'black', 'orange', 'orange'])
    axes[1].set_ylabel('Fehler auf Testdaten')
    axes[1].set_title(f'Test R2 manuell={manual_test_metrics[2]:.2f} | gelernt={learned_test_metrics[2]:.2f}')
    plt.tight_layout()
    plt.show()

widgets.interact(
    line_fit_demo,
    samples=widgets.IntSlider(value=140, min=40, max=300, step=10),
    m=widgets.FloatSlider(value=10.0, min=0.0, max=25.0, step=0.2),
    b=widgets.FloatSlider(value=350.0, min=-500.0, max=1500.0, step=25.0),
    rauschen=widgets.IntSlider(value=85, min=0, max=220, step=10),
    ausreisser=widgets.IntSlider(value=0, min=0, max=20),
    ausreisser_staerke=widgets.IntSlider(value=700, min=-1200, max=1800, step=100),
);


## Metriken auf derselben einstellbaren Gerade

Hier bleibt die wahre Zielgerade fest. Du verschiebst die Vorhersagegerade mit `m` und `b`. Der große Fehler ist ein fester Zusatzfehler auf einem einzelnen Sample. Seine Größe hängt nicht von `m` oder `b` ab. So wird sichtbar, worauf MAE, RMSE und R² jeweils reagieren.


In [ ]:
def metric_line_demo(samples=30, m=1.8, b=-0.5, grosser_fehler=0):
    x_values = np.linspace(-3, 3, samples)
    y_true = 1.8 * x_values - 0.5
    y_line = predict_manual_line(x_values, m, b)
    y_pred = y_line.copy()
    # Der große Fehler ist bewusst unabhängig von m und b:
    # Dieser eine Vorhersagepunkt liegt exakt grosser_fehler über/unter dem echten Wert.
    y_pred[-1] = y_true[-1] + grosser_fehler

    errors = y_pred - y_true
    mae, rmse, r2 = regression_metrics(y_true, y_pred)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].scatter(x_values, y_true, label='echte Werte')
    axes[0].plot(x_values, y_true, color='green', linestyle='--', label='wahre Funktion')
    axes[0].plot(x_values, y_line, color='black', linewidth=2, label=f'Vorhersagegerade: y={m:.1f}x+{b:.1f}')
    if grosser_fehler != 0:
        axes[0].scatter([x_values[-1]], [y_pred[-1]], color='red', s=90, label=f'fester Fehler: {grosser_fehler:+.0f}')
    axes[0].set_title(f'R2={r2:.2f}')
    axes[0].legend()

    axes[1].bar(range(len(errors)), np.abs(errors))
    axes[1].set_title(f'Absolute Fehler | MAE={mae:.2f}')
    axes[1].set_xlabel('Sample')
    axes[1].set_ylabel('|Fehler|')

    axes[2].bar(range(len(errors)), errors**2)
    axes[2].set_title(f'Quadrierte Fehler | RMSE={rmse:.2f}')
    axes[2].set_xlabel('Sample')
    axes[2].set_ylabel('Fehler^2')
    plt.tight_layout()
    plt.show()

widgets.interact(
    metric_line_demo,
    samples=widgets.IntSlider(value=30, min=6, max=120, step=2),
    m=widgets.FloatSlider(value=1.8, min=-1.0, max=4.0, step=0.1),
    b=widgets.FloatSlider(value=-0.5, min=-5.0, max=5.0, step=0.1),
    grosser_fehler=widgets.IntSlider(value=0, min=-30, max=30, step=1),
);
